In [1]:
import os, sys
import numpy as np
import pandas as pd
import re
import pickle
from sklearn.metrics import roc_auc_score, f1_score
import random
from datetime import date
import time
from sklvq import GLVQ
import sklvq
parts=os.getcwd().split('/')[:-1]
datapath='/'.join(parts)+'/data/'
dname='diabetes'
#dname='breastcancer'
#trainset, testset=pd.read_csv(datapath+'%s_trainset.csv'%dname), pd.read_csv(datapath+'%s_testset.csv'%dname)
#unlearnset=pd.read_csv(datapath+'unlearn_samples.csv')
#features=trainset.columns[:-1]
code_path='/'.join(parts)+'/scripts/'
sys.path.append(code_path)
from experiment_utils import dataset_health
Xtrain, Ytrain, Xtest, Ytest, features=dataset_health(dname)
%load_ext autoreload
#parts=os.getcwd().split('/')
#code_path='/'.join(parts[:parts.index("toolboxes")+1])+'/sklvq/'
#sys.path.append(code_path)
#from sklvq import GMLVQ, LGMLVQ

/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq


## Training of original training set

### Data normalization



In [2]:
%autoreload 2
from experiment_utils import data_normalization, data_norm_log

#zXtrain, zXtest=data_normalization(Xtrain, Xtest)
zXtrain, zXtest=data_norm_log(Xtrain, Xtest)
print(zXtrain.shape, zXtest.shape, np.unique(Ytrain))
#print(Xtrain.isnull().sum())

(81412, 43) (20354, 43) [0 1]


### Training Generalized LVQ models on the original training get 
- A copy of it is also created to measure deviation from original model after unlearning has been accomplished
- Set seed to ensure the copy of original that is made is a true identical copy, for reproducibility and comparability

In [3]:
dist_name, activation_type="squared-euclidean", "identity"
solver_type="lbfgs" #"sgd"
solver_params={"max_runs": 5, "step_size": np.array([0.05]),# "k": 3,
              }
nprots_per_class=2
        
if solver_type in ["sgd", "wgd"]:
    glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
else:
    glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, #solver_params={"callback": lbfgs_callback}, 
        random_state=42)
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, #solver_params={"callback": lbfgs_callback} ,
        random_state=42)

glvq.fit(zXtrain, Ytrain)
glvq_copy.fit(zXtrain, Ytrain)
glvq_copy.secure_prototypes_=glvq.prototypes_.copy()
est_train, est_test=glvq.predict(zXtrain), glvq.predict(zXtest)
est_train_copy, est_test_copy=glvq_copy.predict(zXtrain), glvq_copy.predict(zXtest)

### Samples to unlearn, relearn, and normalization of samples to relearn

- Samples to be unlearned are selected as follows
     * Randomly selected 'n' samples, where n $\in [ 1,5,10,20,30 ]$
     * Select `n' samples that are most distant from eaither or both prototypes

- Relearning is retraining of model on original data minus the samples to be unlearned


### Random selection of 'n' samples to unlearn

In [4]:
%autoreload 2
from utils import samples_unlearn_random, samples_enforce_random
n=1000
random_learn_set=samples_unlearn_random(Xtrain,Ytrain, n,0)
unlearn_indices,relearn_indices=random_learn_set['unlearn_indices'], random_learn_set['relearn_indices']
unlearn_samples,relearn_samples=random_learn_set['unlearn_samples'], random_learn_set['relearn_samples']
unlearn_labs,relearn_labs=random_learn_set['unlearn_labs'], random_learn_set['relearn_labs']
#enforce_samples,enforce_labs=random_learn_set['enforce_samples'], random_learn_set['enforce_samples']
enforce_indices, enforce_samples=samples_enforce_random(Xtrain, unlearn_indices,Ytrain)
zXretrain=zXtrain.iloc[relearn_indices]#, Xtest)
#zXretrain, zXretest=(relearn_samples-mu_new)/std_new, (Xtest-mu_new)/std_new


### Retraining on partial data (only relearn_samples) and unlearning from original trained model

In [5]:
%autoreload 2
from unlearning.unlearn_eval import *
# Retraining of GLVQ model with all params excatly the same as teh original model
dev0, max_dev_indx0=compare_fidelity_glvq(glvq, glvq_copy)
print('Before unlearning: Deviation between original model and its copy:', dev0)
st=time.time()
glvq_partial1=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type#, solver_params=solver_params
)
glvq_partial1.fit(zXretrain, relearn_labs)
elapsed_retrain=(time.time()-st)/60
est_partial_retrain, est_partial_retest=glvq_partial1.predict(zXretrain), glvq_partial1.predict(zXtest)


Before unlearning: Deviation between original model and its copy: 0.0


In [6]:
zXtrain.head(4)
print(glvq.prototypes_[0])
print(glvq_partial1.prototypes_[0])

[-3.93552203e+00 -5.29779865e+00 -1.03987766e-04  8.70919910e-01
  3.29287836e+00 -1.61335405e+00  2.44102728e+00 -1.06441468e+01
 -1.10451908e+01 -1.47917000e+01  1.58582163e+00 -9.23968448e+00
 -9.21179151e+00  3.96752997e+00 -9.21022415e+00 -9.21034038e+00
 -9.21034037e+00 -8.60234838e+00 -9.21154630e+00 -8.32682334e+00
 -9.21215095e+00 -7.87980840e+00 -9.11891544e+00 -3.62621168e+00
 -5.61795530e+00 -9.20907953e+00 -9.20757880e+00 -9.21811089e+00
 -9.15405797e+00 -8.59060650e+00 -9.23028569e+00 -8.67662170e+00
 -9.20354687e+00 -9.20291444e+00 -9.21154769e+00 -8.94131092e+00
 -1.86544974e+00 -9.16131195e+00 -8.76654838e+00 -8.10674050e+00
 -7.39392751e+00 -3.35570639e+00 -8.52865361e+00]
[ -4.83677789  -9.21128204 -10.31848507   0.83397845   3.29783533
  -2.53388129   2.12821916 -10.23524302 -10.30718905 -12.50956066
   1.7145448   -9.21511251  -9.21034032   4.0881898   -9.20446428
  -9.21034037  -9.21034037  -9.31249986  -9.21034013  -9.55958251
  -9.21172219  -9.3643322   -9.24974

### Unlearning from copy of original model, and comparing model characteristics and performance
* Characteristics comparison: Fidelity of prototypes
* Performance comparison: Raw deviation in predictions, deviation in Balanced accuracy and ROC-AUC

In [28]:
%autoreload 2
from collections import Counter
from unlearning.unlearn_eval import compare_perf
from unlearning.unlearn_lvq import unlearn_sample_effect_glvq, unlearn_relearn_sample_glvq
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score, root_mean_squared_error
from unlearning.unlearn_eval import compare_fidelity_glvq

training_info={'setsize':zXtrain.shape[0], 'class_weight':Counter(Ytrain)}
glvq_copy.prototypes_=glvq_copy.secure_prototypes_.copy()
#glvq_copy.prototypes_=glvq.prototypes_.copy()
st_un=time.time()
glvq_copy.unlearn_rate_=0.001
#glvq_partial1.normalize_variables(glvq_partial1.prototypes_)
updated_model=unlearn_relearn_sample_glvq(glvq_copy, zXtrain.iloc[unlearn_indices], unlearn_labs, 
                                          zXtrain.iloc[enforce_indices], Ytrain[enforce_indices],
                                          training_info)
elapsed_untrain=(time.time()-st_un)/60
#glvq_copy.set_prototypes(updated_prots
dev02, max_dev_indx02=compare_fidelity_glvq(glvq, updated_model)
print('After unlearning: Deviation between original model and its unlearned copy:', dev02, 
      np.round(root_mean_squared_error(updated_model.prototypes_, glvq_partial1.prototypes_),5))

dev12, max_dev_indx12=compare_fidelity_glvq(glvq_partial1, updated_model)
print('After unlearning: Deviation between retrained and unlearned moodels:', dev12)

dev01, max_dev_indx01=compare_fidelity_glvq(glvq, glvq_partial1)
print('After retraining: Deviation between original and retrained model:', dev01)

cratio_ur_uo=dev12/dev02
print('dev(prots from retrained and unlearned models) /dev(prots from original and unlearned models)=%0.03f'%cratio_ur_uo)
data_dict={'zX_M1':zXtest}
perf01=compare_perf(glvq, glvq_partial1, data_dict,Ytest)


#model.normalize_variables(model.prototypes_)
print(elapsed_retrain, elapsed_untrain)

After unlearning: Deviation between original model and its unlearned copy: 7.0291311691731675 7.07005
After unlearning: Deviation between retrained and unlearned moodels: 7.070052476146613
After retraining: Deviation between original and retrained model: 0.6331463031799472
dev(prots from retrained and unlearned models) /dev(prots from original and unlearned models)=1.006
0.16554967959721884 0.0044833898544311525


In [25]:
print(rmse(updated_model.prototypes_, glvq_partial1.prototypes_))
print(rmse(updated_model.prototypes_, glvq.prototypes_))
print(rmse(glvq.prototypes_, glvq_partial1.prototypes_))
glvq_partial1.prototypes_[0]

7.070052503194231
7.0291311695840015
0.6331463031799472


array([ -4.83677789,  -9.21128204, -10.31848507,   0.83397845,
         3.29783533,  -2.53388129,   2.12821916, -10.23524302,
       -10.30718905, -12.50956066,   1.7145448 ,  -9.21511251,
        -9.21034032,   4.0881898 ,  -9.20446428,  -9.21034037,
        -9.21034037,  -9.31249986,  -9.21034013,  -9.55958251,
        -9.21172219,  -9.3643322 ,  -9.24974528,  -9.21777922,
        -9.2588623 ,  -9.21034042,  -9.21034051,  -9.20821917,
        -9.22138342,  -9.36640035,  -9.29318883,  -9.29561322,
        -9.20853536,  -9.20301876,  -9.21033988,  -8.14285399,
        -2.60188578,  -9.18976655,  -8.60024789,  -8.30660728,
        -8.00324204,  -2.0775501 ,  -8.72309114])

In [30]:
clabs=Counter(unlearn_labs)
for c in clabs.keys():
    print(clabs[c])

#glvq.get_variables().shape

462
538


In [32]:
deviation02=rmse(glvq.prototypes_,updated_model.prototypes_)
deviation01=rmse(glvq.prototypes_,glvq_partial1.prototypes_)
deviation12=rmse(glvq_partial1.prototypes_,updated_model.prototypes_)
dev02, max_dev_indx02=compare_fidelity_glvq(glvq, updated_model)
print(deviation02, dev02, deviation01, dev01, deviation12, dev12)


1.3221922495896087e-09 1.3221922495896087e-09 0.009563674451934052 0.009563674451934057 0.009563674452122892 0.009563674452122892


In [15]:
from sklearn.metrics.cluster import adjusted_mutual_info_score as amis
from sklearn.metrics import root_mean_squared_error as rmse
# example of calculating the kl divergence between two mass functions
#from math import log2
 
# calculate the kl divergence
def kl_divergence(p, q):
	return sum(p[i] * np.log(p[i]/q[i])/np.log(2) for i in range(len(p)))

from scipy.stats import entropy   
print(normalized_mutual_info_score(glvq_partial1.predict(zXtrain),updated_model.predict(zXtrain)),#[:,1]),
       normalized_mutual_info_score(glvq_partial1.predict_proba(zXtrain)[:,1],glvq.predict_proba(zXtrain)[:,1]),
     normalized_mutual_info_score(updated_model.predict_proba(zXtrain)[:,1],glvq.predict_proba(zXtrain)[:,1]))
for i in range(0, updated_model.prototypes_.shape[0]):
  #  print(kl_divergence(glvq_partial1.prototypes_[i], updated_model.prototypes_[i]),
  #        kl_divergence(glvq_partial1.prototypes_[i], glvq.prototypes_[i]),
  #        kl_divergence(updated_model.prototypes_[i], glvq.prototypes_[i]))
     print( normalized_mutual_info_score(glvq_partial1.prototypes_[i],glvq.prototypes_[i]),
        normalized_mutual_info_score(glvq_partial1.prototypes_[i],updated_model.prototypes_[i]),
         normalized_mutual_info_score(glvq.prototypes_[i],updated_model.prototypes_[i]),
           rmse(glvq_partial1.prototypes_[i],updated_model.prototypes_[i])
         #normalized_mutual_info_score(glvq_partial1.predict(zXtrain),updated_model.predict(zXtrain))
         )
    

/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete 

0.5894665222955001 1.0 1.0
1.0 1.0 1.0 0.0008251385517378058
1.0 1.0 1.0 0.00043646475077164306
1.0 1.0 1.0 0.027677902554655957
1.0 1.0 1.0 0.046559743568228555


/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete 

In [9]:
print(glvq_partial1.prototypes_[1])
print(updated_model.prototypes_[1])
print(normalized_mutual_info_score(glvq_partial1.predict_proba(zXtrain)[:,1],updated_model.predict_proba(zXtrain)[:,1]))
print(normalized_mutual_info_score(glvq.predict_proba(zXtrain)[:,1],updated_model.predict_proba(zXtrain)[:,1]))

#np.sum([dist_same, dist_diff], axis=0)

[-0.08759795 -0.16714634 -0.18655304  0.01505955  0.05970634 -0.04509102
  0.03871026 -0.18551387 -0.18648478 -0.22692012  0.03108744 -0.16726739
 -0.1671185   0.07416793 -0.16695927 -0.1671185  -0.1671185  -0.16902463
 -0.1671185  -0.17315174 -0.16714372 -0.1694149  -0.16789207 -0.16728056
 -0.16783836 -0.1671185  -0.1671185  -0.16711043 -0.16739881 -0.17005456
 -0.16851102 -0.16878769 -0.1670529  -0.16699581 -0.16711848 -0.14975059
 -0.04618718 -0.16681818 -0.15537865 -0.1503394  -0.14526564 -0.03761973
 -0.158218  ]
[-0.0878336  -0.16708483 -0.18725584  0.01515018  0.05983449 -0.04587094
  0.03862878 -0.18531479 -0.18666166 -0.22727308  0.03114807 -0.16717762
 -0.167069    0.07417024 -0.16691704 -0.167069   -0.167069   -0.16893565
 -0.167069   -0.17344841 -0.16709382 -0.16984309 -0.16784472 -0.16719337
 -0.16793753 -0.167069   -0.167069   -0.16703144 -0.1672691  -0.16996888
 -0.16846529 -0.16867115 -0.16700428 -0.16697417 -0.167069   -0.1476753
 -0.04735291 -0.1666656  -0.15605069 -

/home/sreejita/miniconda3/envs/pyenv_2026/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:66: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and continuous values for target
  warnings.warn(msg, UserWarning)


1.0


In [35]:
%autoreload 2
from utils import relearn_unlearn_samples
dist_name, activation_type="squared-euclidean", "identity"
solver_type, solver_params="sgd", {"max_runs": 5, "step_size": np.array([0.05]),  #"k": 3,
                                  }
nprots_per_class=2
glvq=model = GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)

glvq.fit(zXtrain, Ytrain)
training_info={'setsize':zXtrain.shape[0], 'class_weight':Counter(Ytrain)}
########################################################################################
dist_func=sklvq.distances.SquaredEuclidean()
rel_distances=dist_func(zXtrain, glvq)
normed_distances=rel_distances/rel_distances.sum(axis=1)[:,np.newaxis]
sorted_dist, sorted_ind=np.sort(normed_distances, axis=1), np.argsort(normed_distances, axis=1)
closest_dists=(sorted_dist[:,1] -sorted_dist[:,0])
#thresh_opts=[0.000001,0.00001, 0.0001,0.001,0.01]
if dname=='criteo':
    nopts=[0.0001,0.001,0.01,0.1]
else:
    nopts=[0.0001,0.001,0.01,0.05,0.1]
########################################################################################
# Unlearning parameters to compare
# * number of random samples to unlearn n=[1,5,20,30,50]
cint=0
for n in nopts:
    unlearn_indices=np.where(closest_dists<=n)[0]
    relearn_indices, relearn_samples=relearn_unlearn_samples(Xtrain, unlearn_indices,0)
    if (len(unlearn_indices)==0)|(len(relearn_indices)==0):
        print(n, len(unlearn_indices))
        continue
    glvq_copy=GLVQ(#force_all_finite="allow-nan",
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy.fit(zXtrain, Ytrain)
    dev00, max_dev_indx0=compare_fidelity_glvq(glvq, glvq_copy)
    print('Before unlearning: Deviation between original model and its copy:', dev00)
    #outlier_learn_set=samples_unlearn_outliers(zXtrain,glvq, n,0)
    outlier_learn_set={'unlearn_indices':unlearn_indices, 'unlearn_samples':Xtrain.iloc[unlearn_indices],
                 'relearn_indices':relearn_indices, 'relearn_samples':relearn_samples}
   # outlier_learn_set['relearn_labs']=Ytrain[outlier_learn_set['relearn_indices']]
   # outlier_learn_set['unlearn_labs']=Ytrain[outlier_learn_set['unlearn_indices']]
    #outlier_learn_set=samples_unlearn_outliers(Xtrain,Ytrain, n,0)
    unlearn_indices,relearn_indices=outlier_learn_set['unlearn_indices'], outlier_learn_set['relearn_indices']
    unlearn_samples,relearn_samples=outlier_learn_set['unlearn_samples'], outlier_learn_set['relearn_samples']
    unlearn_labs,relearn_labs=Ytrain[unlearn_indices], Ytrain[relearn_indices]
    zXretrain, zXretest=data_norm_log(Xtrain.iloc[relearn_indices], Xtest)
    #Retraining 
    st=time.time()
    ####################################
    glvq_partial1=GLVQ(distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
        solver_type=solver_type, solver_params=solver_params)
    glvq_partial1.fit(zXretrain, relearn_labs)
    #####################################
    elapsed_retrain=(time.time()-st)/60
    #####################################
    dev01, max_dev_indx01=compare_fidelity_glvq(glvq, glvq_partial1)
    print('After retraining: Deviation between original and retrained models:', dev01)
    ###############################################################################################################
    #Unlearning
    st_un=time.time()
    #########################################
    updated_model=unlearn_sample_effect_glvq(glvq_copy, zXtrain.iloc[unlearn_indices], unlearn_labs, training_info)
    #########################################
    elapsed_untrain=(time.time()-st_un)/60
    print('n=%d, Elapsed time diff=%3f-%3f'%(len(unlearn_indices), elapsed_retrain,elapsed_untrain))
    #########################################
    dev02, max_dev_indx02=compare_fidelity_glvq(glvq, updated_model)
    print('After unlearning: Deviation between original and unlearned models:', dev02)
    dev12, max_dev_indx12=compare_fidelity_glvq(glvq_partial1,updated_model)
    print('After unlearning: Deviation between retrained and unlearned models:', dev12)
    ###############################################################################################################
    cratio_uo_ur=dev02/dev12
    print('Dev(prots from original and unlearned models)/Dev(prots from retrained and unlearned models)=%0.03f'%cratio_uo_ur)
    ###############################################################################
    # Compare original (0) vs retrained (1)
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf01=compare_perf(glvq, glvq_partial1, data_dict,Ytest)
    # Compare original (0) vs unlearned (2)
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf02=compare_perf(glvq, updated_model, data_dict,Ytest)
    # Compare retrained (1) vs vs unlearned (2)
    data_dict={'zX_M1':zXretest,'zX_M2':zXretest}
    perf12=compare_perf(glvq_partial1, updated_model, data_dict,Ytest)
    ##############################################################################
    compare_dict={'num_prot': nprots_per_class, 'n':len(unlearn_indices), 'Mapping':'0:original; 1:retrain; 2:unlearn',
        'et_retrain':elapsed_retrain, 'et_unlearn':elapsed_untrain,
        'prot_dev_01': dev01,'prot_dev_02': dev02,'prot_dev_12': dev12, 
        'prot_dev_02by12':cratio_uo_ur, #'prot_dev_02':1-dev02,  'prot_dev_12':1-dev12,
        'dev_nAcc_01': perf01['dev_npreds'],  'dev_nAcc_02': perf02['dev_npreds'],  'dev_nAcc_12': perf12['dev_npreds'],
        'Acc_M0': perf01['M1_acc'], 'Acc_M1': perf12['M1_acc'], 'Acc_M2': perf12['M2_acc'], 
        'AUC_M0': perf01['M1_auc'], 'AUC_M1': perf12['M1_auc'], 'AUC_M2': perf12['M2_auc'], 
       # 'dev_auc_01': perf01['dev_auc'], 'dev_auc_02': perf02['dev_auc'], 'dev_auc_12': perf12['dev_auc']
                 }
    if cint==0: #'dev_auc_M1M2'
        compare_df=pd.DataFrame.from_dict(data=compare_dict, orient='index').T
        cint+=1
    else:
        temp= pd.DataFrame.from_dict(data=compare_dict, orient='index').T
        compare_df=pd.concat([compare_df, temp])
        print(temp)
        cint+=1

0.0001 0
0.001 0
0.01 11463263
0.1 11463263


In [45]:
nopts=[0.001, 0.0002, 0.003, 0.005]
print(dname)
print(np.min(closest_dists), np.median(closest_dists),np.max(closest_dists))
for n in nopts:
    unlearn_indices=np.where(closest_dists<=n)[0]
    relearn_indices, relearn_samples=relearn_unlearn_samples(Xtrain, unlearn_indices,0)
    if (len(unlearn_indices)==0)|(len(relearn_indices)==0):
        print(n, len(unlearn_indices), len(relearn_indices))

criteo
0.002100861989661666 0.0024557131718477396 0.002909624378193771
0.001 0 11463263
0.0002 0 11463263
0.003 11463263 0
0.005 11463263 0


In [19]:
print(compare_df)
resultspath='/'.join(parts)+'/results/'
compare_df.to_csv(resultspath+'%s_outlier_unlearn_retrain_compare_nprot%d%s.csv'%(dname,nprots_per_class, solver_type), index=False, sep='\t')

  num_prot      n                           Mapping et_retrain et_unlearn  \
0        2     10  0:original; 1:retrain; 2:unlearn   0.222089   0.000226   
0        2    127  0:original; 1:retrain; 2:unlearn   0.314243   0.000381   
0        2  15166  0:original; 1:retrain; 2:unlearn     0.1938   0.000593   

  prot_dev_01 prot_dev_02 prot_dev_12 prot_dev_02by12 dev_nAcc_01 dev_nAcc_02  \
0     0.70336     5.75302     5.77789        0.995696        10.0       356.0   
0     0.54938        5.75     5.76861        0.996774         2.0        49.0   
0     1.72667     5.73147     5.49481         1.04307       113.0        33.0   

  dev_nAcc_12  Acc_M0  Acc_M1  Acc_M2  AUC_M0  AUC_M1  AUC_M2  
0       346.0  0.5855  0.5838  0.5269  0.5931  0.5935  0.6905  
0        47.0  0.5855  0.5849  0.5811  0.5931  0.5934  0.6977  
0       -80.0  0.5855  0.5854  0.5874  0.5931  0.6421  0.7233  


In [22]:
dist_func=sklvq.distances.SquaredEuclidean()
rel_distances=dist_func(zXtrain, glvq)
normed_distances=rel_distances/rel_distances.sum(axis=1)[:,np.newaxis]
sorted_dist, sorted_ind=np.sort(normed_distances, axis=1), np.argsort(normed_distances, axis=1)
closest_dists=(sorted_dist[:,1] -sorted_dist[:,0])
thresh_opts=[0.35,0.38, 0.4, 0.42, 0.44]
for thresh in thresh_opts:
    unlearn_indices2=np.where(closest_dists<=thresh)[0]
    print(thresh, len(Ytrain), len(unlearn_indices2))
    if thresh==0.4:
        print(rel_distances[unlearn_indices2,:])
    unlearn_samples2=zXtrain.iloc[unlearn_indices2].copy()
    outlier_learn_set2=samples_unlearn_outliers(zXtrain,glvq, n,0)
    
rel_distances[unlearn_indices2,:]

0.35 81412 81412


NameError: name 'samples_unlearn_outliers' is not defined

In [24]:
closest_dists

array([0.00570646, 0.02083404, 0.00168147, ..., 0.00038188, 0.00078343,
       0.0046479 ])

In [95]:
rel_distances

array([[  12.59533194, 1593.54611123, 1593.54267533],
       [   8.2208417 , 1614.84381882, 1614.8400971 ],
       [  32.14440009, 1567.15114384, 1567.14938803],
       ...,
       [   6.66853435, 1641.59708184, 1641.59385068],
       [  45.13539977, 1735.94034003, 1735.93769355],
       [  20.77454422, 1426.24194148, 1426.239213  ]])

In [21]:
%autoreload 2
from utils import samples_unlearn_random
from unlearning.unlearn_eval import *
from unlearning.unlearn_lvq import unlearn_sample_effect_glvq

dist_name, activation_type="squared-euclidean", "identity"
solver_type, solver_params="wgd", {"max_runs": 5, "k": 3, "step_size": np.array([0.05])}
nprots_per_class=3
glvq=model = GLVQ(
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)

glvq.fit(zXtrain, Ytrain)

########################################################################################
# Unlearning parameters to compare
# * number of random samples to unlearn n=[1,5,20,30,50]

for n in [1,5,20,30,50,100]:
    glvq_copy=GLVQ(
    distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
    solver_type=solver_type, solver_params=solver_params,random_state=42)
    glvq_copy.fit(zXtrain, Ytrain)
    dev00, max_dev_indx0=compare_fidelity_glvq(glvq, glvq_copy)
    print('Before unlearning: Fidelity between original model and its copy:', 1-dev00)
    random_learn_set=samples_unlearn_random(Xtrain,Ytrain, n,0)
    unlearn_indices,relearn_indices=random_learn_set['unlearn_indices'], random_learn_set['relearn_indices']
    unlearn_samples,relearn_samples=random_learn_set['unlearn_samples'], random_learn_set['relearn_samples']
    unlearn_labs,relearn_labs=random_learn_set['unlearn_labs'], random_learn_set['relearn_labs']
    zXretrain, zXretest=data_normalization(Xtrain.iloc[relearn_indices], Xtest)
    st=time.time()
    glvq_partial1=GLVQ(distance_type=dist_name, activation_type=activation_type, prototype_n_per_class=nprots_per_class,
        solver_type=solver_type, solver_params=solver_params)
    glvq_partial1.fit(zXretrain, relearn_labs)
    elapsed_retrain=(time.time()-st)/60
    st_un=time.time()
    updated_prots=unlearn_sample_effect_glvq(glvq, zXtrain.iloc[unlearn_indices], unlearn_labs)
    elapsed_untrain=(time.time()-st_un)/60
    glvq_copy.set_prototypes(updated_prots)
    dev01, max_dev_indx01=compare_fidelity_glvq(glvq, glvq_copy)
    dev12, max_dev_indx12=compare_fidelity_glvq(glvq_partial1,glvq_copy)
    dev02, max_dev_indx02=compare_fidelity_glvq(glvq, glvq_copy)
    print('After unlearning: Fidelity between retrained and unlearned moodels:', 1-dev12)
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf01=compare_perf(glvq, glvq_partial1, data_dict,Ytest)
    
    data_dict={'zX_M1':zXtest,'zX_M2':zXretest}
    perf02=compare_perf(glvq, glvq_copy, data_dict,Ytest)
    
    data_dict={'zX_M1':zXretest,'zX_M2':zXretest}
    perf12=compare_perf(glvq_partial1, glvq_copy, data_dict,Ytest)
    compare_dict={'n':n,'Mapping':'0:original; 1:retrain; 2:unlearn',
        'prot_dev_01':1-dev01, 'prot_dev_02':1-dev02,  'prot_dev_12':1-dev12,
        'dev_npreds_01': perf01['dev_npreds'],  'dev_npreds_02': perf02['dev_npreds'],  'dev_npreds_12': perf12['dev_npreds'],
        'dev_acc_01': perf01['dev_acc'], 'dev_acc_02': perf02['dev_acc'], 'dev_acc_12': perf12['dev_acc'], 
        'dev_auc_01': perf01['dev_auc'], 'dev_auc_02': perf02['dev_auc'], 'dev_auc_12': perf12['dev_auc']}
    print('n=%d, time diff=%3f-%3f'%(n, elapsed_retrain,elapsed_untrain))
    if n==1: #'dev_auc_M1M2'
        compare_df=pd.DataFrame.from_dict(data=compare_dict, orient='index').T
    else:
        compare_df=pd.concat([compare_df, 
                             pd.DataFrame.from_dict(data=compare_dict, orient='index').T])
compare_df.insert(0,'num_prot', nprots_per_class)

compare_df

Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.9570325405058471
n=1, time diff=0.031552-0.000383
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.955020863006206
n=5, time diff=0.030930-0.000469
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.88257557988577
n=20, time diff=0.031545-0.000615
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.8626778575086758
n=30, time diff=0.032176-0.000745
Before unlearning: Fidelity between original model and its copy: 1.0
After unlearning: Fidelity between retrained and unlearned moodels: 0.8868824043623249
n=50, time diff=0.027314-0.000787
Before unlearning: Fidelity between original model and 

,num_prot,n,Mapping,prot_dev_01,prot_dev_02,prot_dev_12,dev_npreds_01,dev_npreds_02,dev_npreds_12,dev_acc_01,dev_acc_02,dev_acc_12,dev_auc_01,dev_auc_02,dev_auc_12
0,3,1,0:original; 1:retrain; 2:unlearn,0.979945,0.979945,0.957033,4696,4339,6003,0.000579,-0.002905,-0.003484,0.00418,0.007115,0.002935
0,3,5,0:original; 1:retrain; 2:unlearn,0.981692,0.981692,0.955021,4624,3481,6713,0.001488,-0.001968,-0.003456,0.003091,0.002451,-0.00064
0,3,20,0:original; 1:retrain; 2:unlearn,0.899237,0.899237,0.882576,3510,17456,18027,0.000828,0.018603,0.017776,0.001388,0.024831,0.023443
0,3,30,0:original; 1:retrain; 2:unlearn,0.873257,0.873257,0.862678,4569,17315,19048,0.000527,0.010208,0.009681,0.003171,0.018993,0.015822
0,3,50,0:original; 1:retrain; 2:unlearn,0.900429,0.900429,0.886882,4302,13704,14113,0.000757,0.012609,0.011851,0.001273,0.007967,0.006694
0,3,100,0:original; 1:retrain; 2:unlearn,0.935408,0.935408,0.915804,5201,10781,12983,-0.000243,0.007323,0.007566,0.004353,0.001282,-0.003071


In [8]:
a=3.09678504
np.round(a, 3)

3.097

In [9]:
np.round(a, 4)

3.0968